# SmartAd Performance Analysis — Exploratory Data Analysis

---

> **Notebook Purpose:** End-to-end exploratory analysis of the SmartAd A/B testing dataset to understand campaign performance, audience behaviour, and brand awareness lift across experimental groups.

**Author:** SmartAd Analytics Team  
**Last Updated:** August 2026  
**Dataset:** AdSmart A/B Testing Data (`AdSmart_AB_testing_data.csv`)

## 1. Business Context

### Problem Description

SmartAd is a mobile-first advertising agency that designs and delivers targeted digital ad campaigns for its clients. The agency ran a **brand-awareness A/B test** to evaluate whether showing a specific advertisement increases consumer awareness of a product relative to a control group that saw a Public Service Announcement (PSA) or no ad at all.

Understanding which segments respond best to ads — by device, browser, time-of-day, and date — enables the marketing team to:
- **Allocate budget** more efficiently toward high-converting audiences.
- **Optimise ad scheduling** to peak engagement hours.
- **Report ROI** to the client with statistical rigour.

---

### Campaign Goal

| Metric | Target |
|---|---|
| Primary KPI | Brand-awareness rate (% of respondents who answered **Yes** to brand recall) |
| Secondary KPI | Response rate (% of users who actually responded to the survey) |
| Success Criterion | Statistically significant uplift in the **exposed** group vs **control** group |

---

### Hypothesis

| | Statement |
|---|---|
| **H₀ (Null)** | The advertisement has **no effect** on brand awareness — the conversion rates of the exposed and control groups are equal. |
| **H₁ (Alternative)** | The advertisement **increases** brand awareness — the exposed group has a significantly higher conversion rate than the control group. |

> **Significance level:** α = 0.05 (two-tailed test)

## 2. Dataset Schema

The dataset contains one row per ad auction impression. The columns are described below:

| Column | Data Type | Description |
|---|---|---|
| `auction_id` | string (UUID) | Unique identifier for each ad auction / impression event |
| `experiment` | string (categorical) | Group assignment — `'exposed'` (saw the SmartAd) or `'control'` (saw a PSA) |
| `date` | string / date | Calendar date the impression was recorded (YYYY-MM-DD) |
| `hour` | integer (0–23) | Hour of the day (24-hour clock) the impression occurred |
| `device_make` | string | Manufacturer / model of the user's device (e.g., Samsung, Apple iPhone) |
| `platform_os` | integer | Numeric code representing the user's operating system |
| `browser` | string | Web browser used by the user (e.g., Chrome, Safari, Firefox) |
| `yes` | integer (0 or 1) | Survey response — `1` if the user recalled the brand (**aware**), `0` otherwise |
| `no` | integer (0 or 1) | Survey response — `1` if the user did **not** recall the brand, `0` otherwise |

> **Note:** When both `yes == 0` and `no == 0`, the user did **not respond** to the survey and should be excluded from conversion-rate calculations.

## 3. Environment Setup

Import all required libraries. We use:
- **pandas / numpy** — data manipulation and numerical operations
- **matplotlib / seaborn** — static visualisations
- **plotly.express** — interactive visualisations
- **warnings** — suppress non-critical warnings for cleaner output

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Static visualisation ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Interactive visualisation ─────────────────────────────────────────────────
import plotly.express as px

# ── Global plot style ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('All libraries imported successfully ✓')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
print(f'  seaborn : {sns.__version__}')

## 4. Data Loading

Load the raw A/B testing dataset from the `data/` directory. We immediately parse the `date` column to `datetime` and cast `hour` to an integer for easier time-series manipulation later.

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
DATA_PATH = '../data/AdSmart_AB_testing_data.csv'

df = pd.read_csv(DATA_PATH)

# Parse date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Ensure hour is integer
df['hour'] = df['hour'].astype(int)

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(10)

## 5. Basic Dataset Information

We inspect:
1. **Column types and non-null counts** — `df.info()` to catch unexpected nulls or mis-typed columns.
2. **Descriptive statistics** — `df.describe()` for numerical columns.
3. **Shape** — quick sanity check on dataset dimensions.

In [ ]:
# ── Column types & non-null counts ────────────────────────────────────────────
print('=' * 55)
print('DATASET INFO')
print('=' * 55)
df.info()

print('\n')
print('=' * 55)
print('DESCRIPTIVE STATISTICS (numeric columns)')
print('=' * 55)
display(df.describe())

print('\n')
print('=' * 55)
print(f'DATASET SHAPE: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('=' * 55)

## 6. Missing Value Analysis

Missing values can bias downstream analysis. We calculate the count **and percentage** of nulls per column.

- Values that are structurally `0` (such as unanswered survey rows) are **not** missing in the traditional sense — they are handled in Section 8.
- True `NaN` values in categorical columns (e.g., `device_make`, `browser`) may indicate tracking failures.

In [ ]:
# ── Missing value summary ─────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct
}).sort_values('Missing Count', ascending=False)

print('Missing Values per Column:')
print('─' * 40)
display(missing_df)

# Visual bar chart
if missing.sum() > 0:
    ax = missing[missing > 0].plot(
        kind='bar', color='salmon', edgecolor='black', title='Missing Values by Column'
    )
    ax.set_ylabel('Count')
    ax.set_xlabel('Column')
    plt.tight_layout()
    plt.show()
else:
    print('\n✓ No missing values detected in any column.')

## 7. Respondent Analysis — Filtering Non-Responders

### Survey Logic

Each user who saw an ad was shown a one-question survey:
> *"Do you recall seeing this brand?"*

Their response is encoded as:
| `yes` | `no` | Meaning |
|---|---|---|
| 1 | 0 | User **recalled** the brand (aware) |
| 0 | 1 | User **did not** recall the brand |
| 0 | 0 | User **did not respond** to the survey |

Rows where `yes == 0 AND no == 0` represent **non-respondents** — they were served the ad but never answered the question. Including them would **artificially deflate** conversion rates.

We quantify the non-response rate here and create a clean analysis dataset.

In [ ]:
# ── Identify respondents and non-respondents ──────────────────────────────────
total_impressions = df.shape[0]

# Users who answered the survey
respondents_shape = df.query('not (yes==0 and no==0)').shape
# Users who did NOT answer the survey
non_respondents_shape = df.query('yes==0 and no==0').shape

print('─' * 50)
print(f'Total impressions       : {total_impressions:>10,}')
print(f'Survey respondents      : {respondents_shape[0]:>10,}  ({respondents_shape[0]/total_impressions*100:.1f}%)')
print(f'Non-respondents (yes=0, no=0): {non_respondents_shape[0]:>4,}  ({non_respondents_shape[0]/total_impressions*100:.1f}%)')
print('─' * 50)

# Visualise the breakdown
labels = ['Responded', 'Did Not Respond']
sizes  = [respondents_shape[0], non_respondents_shape[0]]
colors = ['#4C72B0', '#DD8452']

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct='%1.1f%%',
    colors=colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for autotext in autotexts:
    autotext.set_fontsize(13)
ax.set_title('Survey Response Rate', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Experiment Group Distribution

A balanced A/B test requires roughly equal group sizes. We verify the split between `exposed` (saw the SmartAd) and `control` (saw a PSA) groups across all impressions.

An imbalanced split could indicate:
- Traffic allocation issues in the ad-serving platform.
- Biased audience targeting.
- Need for stratification in statistical testing.

In [ ]:
# ── Experiment group distribution ─────────────────────────────────────────────
group_counts = df['experiment'].value_counts()
group_pct    = df['experiment'].value_counts(normalize=True).mul(100).round(2)

group_summary = pd.DataFrame({
    'Count': group_counts,
    'Percentage (%)': group_pct
})
print('Experiment Group Distribution:')
display(group_summary)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Count bar
group_counts.plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'],
                  edgecolor='black', width=0.5)
axes[0].set_title('Impression Count by Experiment Group')
axes[0].set_xlabel('Group')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(group_counts.index, rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# Pie chart
axes[1].pie(
    group_counts, labels=group_counts.index,
    autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('Group Split (%)')

plt.suptitle('Experiment Group Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Conversion (Brand Awareness) Rates

We now compute the **core business metric**: the brand-awareness conversion rate for each experiment group.

**Definition:**  
`conversion_rate = (number of users who answered 'yes') / (total survey respondents in the group)`

This section works on the **clean** subset (survey respondents only — users where `yes=0 AND no=0` are excluded).

In [ ]:
# ── Conversion rate calculation ───────────────────────────────────────────────
# Filter to survey respondents only
clean = df.query('not (yes==0 and no==0)').copy()

# 'aware' == 1 means the user recalled the brand (answered yes)
clean['aware'] = clean['yes']

# Aggregate by experiment group
rates = clean.groupby('experiment')['aware'].agg(['mean', 'count', 'sum'])
rates.columns = ['conversion_rate', 'total_respondents', 'successes']
rates['conversion_rate_pct'] = (rates['conversion_rate'] * 100).round(2)

print('Brand Awareness Conversion Rates:')
print('─' * 55)
print(rates.to_string())
print('─' * 55)

# Compute lift
exposed_rate = rates.loc['exposed', 'conversion_rate'] if 'exposed' in rates.index else None
control_rate = rates.loc['control', 'conversion_rate'] if 'control' in rates.index else None

if exposed_rate and control_rate:
    lift = ((exposed_rate - control_rate) / control_rate) * 100
    print(f'\n📈 Relative Lift (exposed vs control): {lift:+.2f}%')

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    rates.index,
    rates['conversion_rate_pct'],
    color=['#4C72B0', '#DD8452'],
    edgecolor='black',
    width=0.4
)
for bar, (_, row) in zip(bars, rates.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{row['conversion_rate_pct']:.2f}%",
        ha='center', va='bottom', fontsize=13, fontweight='bold'
    )
ax.set_title('Brand Awareness Conversion Rate by Experiment Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Experiment Group')
ax.set_ylabel('Conversion Rate (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

## 10. Device Analysis

Understanding which device manufacturers drive the most impressions helps the ad-serving team:
- **Optimise creative formats** for dominant screen sizes (e.g., Samsung flagships vs budget Androids).
- **Detect device-specific rendering issues** that could suppress awareness.

We show the **top 15 device makes** by impression count, split by experiment group.

In [ ]:
# ── Top 15 device makes by impression count ───────────────────────────────────
top15_devices = df['device_make'].value_counts().head(15)

print('Top 15 Device Makes:')
display(top15_devices.to_frame('impression_count'))

# ── Horizontal bar chart ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(
    x=top15_devices.values,
    y=top15_devices.index,
    palette='Blues_d',
    ax=ax,
    edgecolor='grey'
)
ax.set_title('Top 15 Device Makes by Impression Count', fontsize=14, fontweight='bold')
ax.set_xlabel('Impression Count')
ax.set_ylabel('Device Make')
# Annotate bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}',
                (p.get_width() + top15_devices.max() * 0.005, p.get_y() + p.get_height() / 2),
                va='center', ha='left', fontsize=10)
plt.tight_layout()
plt.show()

# ── Split by experiment group ─────────────────────────────────────────────────
device_group = (
    df[df['device_make'].isin(top15_devices.index)]
    .groupby(['device_make', 'experiment'])
    .size()
    .unstack(fill_value=0)
    .loc[top15_devices.index]   # preserve top-15 ordering
)

device_group.plot(kind='barh', figsize=(12, 7), colormap='Set2', edgecolor='grey')
plt.title('Top 15 Device Makes — Impressions Split by Experiment Group', fontsize=14, fontweight='bold')
plt.xlabel('Impression Count')
plt.ylabel('Device Make')
plt.legend(title='Group')
plt.tight_layout()
plt.show()

## 11. Browser Analysis

Browser distribution matters because:
- Different browsers handle **JavaScript and ad formats** differently, which can affect ad render rates and viewability.
- Safari's **Intelligent Tracking Prevention (ITP)** can suppress cookie-based attribution.

We examine the browser mix across all impressions and within each experiment group.

In [ ]:
# ── Browser distribution ──────────────────────────────────────────────────────
browser_counts = df['browser'].value_counts()

print('Browser Distribution:')
display(browser_counts.to_frame('impression_count'))

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# All impressions
sns.barplot(
    x=browser_counts.values,
    y=browser_counts.index,
    palette='Greens_d',
    ax=axes[0],
    edgecolor='grey'
)
axes[0].set_title('Impressions by Browser (All Groups)', fontweight='bold')
axes[0].set_xlabel('Impression Count')
axes[0].set_ylabel('Browser')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_width()):,}',
                     (p.get_width() + browser_counts.max() * 0.01, p.get_y() + p.get_height() / 2),
                     va='center', ha='left', fontsize=10)

# By experiment group
browser_group = (
    df.groupby(['browser', 'experiment'])
    .size()
    .unstack(fill_value=0)
    .loc[browser_counts.index]
)
browser_group.plot(kind='barh', ax=axes[1], colormap='Set2', edgecolor='grey')
axes[1].set_title('Browser Impressions by Experiment Group', fontweight='bold')
axes[1].set_xlabel('Impression Count')
axes[1].set_ylabel('Browser')
axes[1].legend(title='Group')

plt.suptitle('Browser Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Browser awareness rates (clean data) ──────────────────────────────────────
browser_aware = (
    clean.groupby(['browser', 'experiment'])['aware']
    .mean()
    .mul(100)
    .round(2)
    .unstack()
)
print('\nBrand Awareness Rate (%) by Browser and Group:')
display(browser_aware)

## 12. Hourly Trends — Awareness Rate by Hour of Day

Ad recall can vary significantly by time of day due to:
- **Morning commute (7–9 AM):** High mobile usage.
- **Lunch (12–2 PM):** Short browsing sessions.
- **Evening (7–10 PM):** Relaxed, higher dwell time.

We use a **pivot table** to compute the average brand awareness rate for each hour × experiment group combination, then visualise it as a **heatmap**.

In [ ]:
# ── Pivot: hour vs experiment — mean awareness rate ───────────────────────────
hourly_pivot = clean.pivot_table(
    index='hour',
    columns='experiment',
    values='aware',
    aggfunc='mean'
).mul(100).round(2)

print('Hourly Awareness Rate (%) — Pivot Table:')
display(hourly_pivot)

# ── Heatmap ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    hourly_pivot,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Awareness Rate (%)'},
    ax=ax
)
ax.set_title('Brand Awareness Rate (%) by Hour of Day and Experiment Group',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Experiment Group')
ax.set_ylabel('Hour of Day (0 = midnight)')
plt.tight_layout()
plt.show()

# ── Line chart: hourly awareness trend ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
for col in hourly_pivot.columns:
    ax.plot(hourly_pivot.index, hourly_pivot[col], marker='o', label=col)
ax.set_title('Hourly Brand Awareness Rate by Experiment Group', fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Awareness Rate (%)')
ax.set_xticks(range(0, 24))
ax.legend(title='Group')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

## 13. Daily Trends — Awareness Rate Over Time

Tracking conversion rates day-by-day reveals:
- **Learning-effect curves** — whether awareness increases as the campaign matures.
- **Novelty effects** — an initial spike that decays over time.
- **External events** — sudden dips or spikes due to competitor activity or external news.

We plot a **line chart** of daily brand-awareness rate for each experiment group.

In [ ]:
# ── Daily awareness rate ──────────────────────────────────────────────────────
daily_pivot = clean.pivot_table(
    index='date',
    columns='experiment',
    values='aware',
    aggfunc='mean'
).mul(100).round(2)

print('Daily Awareness Rate (%) — Pivot Table:')
display(daily_pivot.head(10))

# ── Static line plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
for col in daily_pivot.columns:
    ax.plot(daily_pivot.index, daily_pivot[col], marker='o', label=col, linewidth=2)
ax.set_title('Daily Brand Awareness Rate by Experiment Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Awareness Rate (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title='Experiment Group')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ── Interactive Plotly line chart ─────────────────────────────────────────────
daily_long = daily_pivot.reset_index().melt(
    id_vars='date', var_name='experiment', value_name='awareness_rate'
)

fig_px = px.line(
    daily_long,
    x='date',
    y='awareness_rate',
    color='experiment',
    markers=True,
    title='Daily Brand Awareness Rate by Experiment Group (Interactive)',
    labels={'awareness_rate': 'Awareness Rate (%)', 'date': 'Date', 'experiment': 'Group'},
    template='plotly_white'
)
fig_px.update_layout(legend_title_text='Experiment Group')
fig_px.show()

## 14. Conclusions & Key Findings

---

### Summary of EDA

| # | Finding | Implication |
|---|---|---|
| 1 | **Non-response rate** — a measurable fraction of users did not answer the survey | Conversion rates must be computed on respondents only; non-response may be non-random |
| 2 | **Group balance** — the exposed and control groups are approximately equal in size | No re-weighting needed for baseline comparisons |
| 3 | **Conversion lift** — the exposed group shows a higher brand-awareness rate than control | Preliminary evidence in favour of H₁; formal statistical test recommended |
| 4 | **Device concentration** — a small number of device makes account for the majority of impressions | Creative optimisation should prioritise the top 5–10 devices |
| 5 | **Browser skew** — most users browse via a dominant browser (e.g., Chrome on Android) | Cross-browser rendering tests are advised for minority browsers |
| 6 | **Hourly patterns** — awareness rates vary by hour, with potential peaks in morning/evening | Ad scheduling can be tuned to high-awareness hours for budget efficiency |
| 7 | **Daily trends** — awareness rates may fluctuate over the campaign period | Monitor for novelty decay; consider refreshing creatives if rates decline |

---

### Recommended Next Steps

1. **Statistical Hypothesis Testing**  
   Run a two-proportion z-test (or chi-squared test) to formally compare exposed vs control conversion rates and compute a p-value and 95% confidence interval.

2. **Sequential Analysis / Early Stopping**  
   Use a sequential probability ratio test (SPRT) to monitor significance during the campaign and stop early if the result is conclusive.

3. **Subgroup / Segment Analysis**  
   Analyse conversion rates broken down by device, browser, and hour to identify **heterogeneous treatment effects** — segments where the ad works best.

4. **Non-Response Bias Assessment**  
   Investigate whether non-respondents differ systematically from respondents (e.g., by device or hour) to understand potential selection bias.

5. **Reporting Dashboard**  
   Productionise key metrics into a live Plotly Dash or Streamlit dashboard for stakeholder monitoring.

---

> *This EDA notebook is the first stage of the SmartAd Performance Analysis pipeline. Findings here feed directly into the `hypothesis_testing.ipynb` and `reporting.ipynb` notebooks.*